In [3]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import warnings

warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv('./used_cars_price_predictor_project/data/pakwheels_data.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26404 entries, 0 to 26403
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          26404 non-null  object
 1   brand         26404 non-null  object
 2   model         26404 non-null  int64 
 3   mileage_km    26404 non-null  int64 
 4   fuel_type     26404 non-null  object
 5   transmission  26404 non-null  object
 6   engine_cc     26404 non-null  int64 
 7   city          26404 non-null  object
 8   price_pkr     26404 non-null  int64 
 9   time          26404 non-null  object
dtypes: int64(4), object(6)
memory usage: 2.0+ MB


In [5]:
df.head()

,name,brand,model,mileage_km,fuel_type,transmission,engine_cc,city,price_pkr,time
0,Toyota Surf 1997 for Sale,Toyota,1997,100000,Petrol,Automatic,2700,Gujrat,4100000,Updated 7 minutes ago
1,Suzuki Jimny 1991 for Sale,Suzuki,1991,130000,Petrol,Manual,1000,Gujrat,1100000,Updated 22 minutes ago
2,Honda City 2004 i-DSI Vario for Sale,Honda,2004,250000,Petrol,Automatic,1300,Lahore,1825000,Updated about 2 hours ago
3,Honda City 2005 i-DSI for Sale,Honda,2005,250000,Petrol,Manual,1300,Lahore,1625000,Updated about 2 hours ago
4,Toyota IST 2003 1.3 A for Sale,Toyota,2003,100000,Petrol,Automatic,1300,Karachi,1500000,Updated about 2 hours ago


In [6]:
X = df.drop(['name', 'time', 'price_pkr'], axis=1)
y = df['price_pkr']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [7]:
numeric_features = X.select_dtypes('int64').columns
categoric_features = X.select_dtypes('object').columns

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categoric_features)
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), categoric_features)
    ]
)

In [9]:
models = {

    'LinearRegression': (
        Pipeline([
            ('preprocessor', preprocessor),
            ('model', LinearRegression())
        ]),
        {}
    ),

    'SVR': (
        Pipeline([
            ('preprocessor', preprocessor),
            ('model', SVR())
        ]),
        {
            'model__kernel': ['rbf', 'poly', 'sigmoid'],
            'model__C': [0.1, 1, 10],
            'model__gamma': [1, 0.1, 0.01],
            'model__epsilon': [0.1, 0.01, 0.001]
        }
    ),

    'DecisionTreeRegressor': (
        Pipeline([
            ('preprocessor', preprocessor_tree),
            ('model', DecisionTreeRegressor())
        ]),
        {
            'model__max_depth': [None, 5, 10],
            'model__splitter': ['best', 'random']
        }
    ),

    'RandomForestRegressor': (
        Pipeline([
            ('preprocessor', preprocessor_tree),
            ('model', RandomForestRegressor())
        ]),
        {
            'model__n_estimators': [10, 100, 1000],
            'model__max_depth': [None, 5, 10]
        }
    ),

    'KNeighborsRegressor': (
        Pipeline([
            ('preprocessor', preprocessor),
            ('model', KNeighborsRegressor())
        ]),
        {
            'model__n_neighbors': np.arange(3, 100, 2),
            'model__weights': ['uniform', 'distance']
        }
    ),

    'XGBRegressor': (
        Pipeline([
            ('preprocessor', preprocessor_tree),
            ('model', XGBRegressor())
        ]),
        {
            'model__n_estimators': [10, 100, 1000],
            'model__learning_rate': [0.1, 0.01, 0.001]
        }
    ),
}

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

name = 'XGBRegressor'
scores = cross_val_score(models[name][0], X, y, cv=kfold)
accuracy = np.mean(scores)
print(f"{name}: {accuracy}")

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\hamza\miniconda3\envs\pythonMLBasics\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\hamza\miniconda3\envs\pythonMLBasics\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\hamza\miniconda3\envs\pythonMLBasics\lib\site-packages\sklearn\ensemble\_forest.py", line 359, in fit
    X, y = validate_data(
  File "c:\Users\hamza\miniconda3\envs\pythonMLBasics\lib\site-packages\sklearn\utils\validation.py", line 2971, in validate_data
    X, y = check_X_y(

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for name, (model, params) in models.items():
    scores = cross_val_score(model, X, y, cv=kfold)
    accuracy = np.mean(scores)
    print(f"{name}: {accuracy}")

LinearRegression: nan
